# SoulIllusions AI Video Maker - GPU Backend (2026-2027 Edition)

This notebook runs a **free GPU server** for text-to-video generation using the **latest 2026-2027 open-source models**.

**Models available (all state-of-the-art):**
- **LTX-Video** (Lightricks) — 768x512, 24fps, ultra-fast, 8GB VRAM
- **Wan 2.2 TI2V-5B** (Alibaba, 2026) — 1280x704, 24fps, unified T2V+I2V, 720P, Apache 2.0
- **Motif-Video 2B** (Motif Technologies, 2026) — 1280x736, 24fps, GGUF quantized, T5Gemma2 encoder
- **Helios-Distilled** (PKU, 2026) — Real-time (19.5 FPS), minute-scale, autoregressive, 6GB VRAM
- **HoloCine** (CVPR 2026) — Multi-shot narrative, persistent character memory, built on Wan2.2

**New Features:**
- Six-dimension structured prompt enhancement
- Post-processing: Super-resolution upscaling + frame interpolation
- GGUF quantization for low-VRAM GPUs
- Audio-ready architecture (MOVA/ID-LoRA compatible)

**Instructions:**
1. Runtime → Change runtime type → T4 GPU (free) or A100/V100 for better models
2. Run all cells in order
3. Copy the URL that appears at the bottom
4. Paste it into the SoulIllusions desktop app

**Tips:**
- LTX-Video is fastest (~30-60s on T4) — use for quick generation
- Wan 2.2 TI2V-5B is best quality (720P@24fps) — use for final quality
- Motif-Video 2B is balanced with GGUF quantization — great for T4
- Helios enables real-time and minute-long generation
- HoloCine for multi-shot narrative scenes

## Step 1: Install Dependencies

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision
!pip install -q omegaconf einops imageio[ffmpeg] imageio-ffmpeg
!pip install -q fastapi uvicorn python-multipart
!pip install -q -U git+https://github.com/huggingface/diffusers
!pip install -q optimum-quanto torchao gguf
!pip install -q nest_asyncio pyngrok
!pip install -q realesrgan basicsr gfpgan
!npm install -g localtunnel

## Step 2: GPU Info & Configuration

In [ ]:
import torch
import numpy as np
import imageio
import os
import time
import uuid
import gc
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {gpu_name} ({vram:.1f} GB VRAM)')
    if vram < 16:
        print('  -> Low VRAM mode: GGUF quantization + CPU offloading enabled')
    elif vram < 24:
        print('  -> Mid VRAM: CPU offloading for large models')
    else:
        print('  -> High VRAM: All models can load directly')

print('\nModels will load on-demand when you generate a video.')
print('\nAvailable models (2026-2027 pipeline):')
print('  LTX-Video:       768x512, 24fps, ultra-fast (~30-60s)')
print('  Wan 2.2 TI2V-5B: 1280x704, 24fps, 720P (~10-20min)')
print('  Motif-Video 2B:  1280x736, 24fps, GGUF (~5-15min)')
print('  Helios-Distilled: Real-time, minute-scale (~2-5s/chunk)')
print('  HoloCine:        Multi-shot narrative, built on Wan2.2')

## Step 3: Prompt Enhancement Engine

Six-dimension structured prompt framework based on 2026 research:
**Subject -> Action -> Frame -> Camera -> Lighting -> Timeline**

In [ ]:
# Six-dimension structured prompt enhancement framework
# Based on 2026 research: Subject -> Action -> Frame -> Camera -> Lighting -> Timeline

STYLE_MODIFIERS = {
    'cinematic': {
        'camera': 'cinematic camera, slow dolly push-in, shallow depth of field',
        'lighting': 'dramatic lighting, golden hour, high contrast, film still',
        'quality': 'movie quality, 4k, highly detailed, professional cinematography',
        'negative': 'static shot, flat lighting, low contrast, amateur, blurry',
    },
    'realistic': {
        'camera': 'handheld camera, natural movement, documentary style',
        'lighting': 'natural lighting, soft ambient light, realistic shadows',
        'quality': 'photorealistic, ultra realistic, 8k, professional photo',
        'negative': 'cartoon, anime, stylized, artificial lighting, oversaturated',
    },
    'anime': {
        'camera': 'dynamic anime camera, expressive angles, smooth panning',
        'lighting': 'vibrant lighting, cel shaded, studio quality lighting',
        'quality': 'anime style, cel shaded, vibrant colors, detailed background, studio quality',
        'negative': 'realistic, photorealistic, dark, muted colors, 3d render',
    },
    'documentary': {
        'camera': 'steady camera, observational, wide establishing shots',
        'lighting': 'natural lighting, available light, realistic',
        'quality': 'documentary style, professional photography, realistic texture',
        'negative': 'stylized, dramatic, artificial, cinematic, music video',
    },
    'music video': {
        'camera': 'dynamic camera movement, fast cuts, tracking shots, crane shots',
        'lighting': 'vibrant colors, dynamic lighting, strobe effects, neon',
        'quality': 'music video aesthetic, stylized, energetic, high production value',
        'negative': 'static, boring, flat, documentary, naturalistic',
    },
    'social media': {
        'camera': 'close-up, selfie angle, vertical composition',
        'lighting': 'bright, colorful, ring light, eye-catching',
        'quality': 'modern, social media style, vibrant, clean',
        'negative': 'dark, cinematic, slow, boring, low quality',
    },
}

NEGATIVE_PROMPT = (
    "worst quality, inconsistent motion, blurry, jittery, distorted, "
    "low resolution, artifacts, static, overexposed, identity drift, "
    "deformation, flickering, ghosting, smearing, duplication, "
    "mutated proportions, inconsistent clothing, flat colors, desaturated"
)

def enhance_prompt(prompt, style='cinematic'):
    """Six-dimension structured prompt enhancement.
    
    Dimensions: Subject -> Action -> Frame -> Camera -> Lighting -> Timeline
    """
    mod = STYLE_MODIFIERS.get(style, STYLE_MODIFIERS['cinematic'])
    
    enhanced = (
        f"{prompt}. "
        f"{mod['camera']}. "
        f"{mod['lighting']}. "
        f"{mod['quality']}. "
        f"Smooth temporal motion, consistent identity throughout. "
        f"Professional grade output."
    )
    
    negative = f"{NEGATIVE_PROMPT}, {mod['negative']}"
    return enhanced, negative

print('Prompt enhancement engine ready!')
print('Six-dimension framework: Subject -> Action -> Frame -> Camera -> Lighting -> Timeline')

## Step 4: Video Generation Functions

All 2026-2027 model loaders and generation functions. Models load on-demand to save VRAM.

In [ ]:
import torch
import os
import time
import uuid
import gc
import numpy as np
import imageio
import subprocess
import warnings
from diffusers.utils import export_to_video
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Lazy-loaded model instances
_pipes = {}

def unload_all_models():
    """Free VRAM by unloading all models."""
    global _pipes
    for key in list(_pipes.keys()):
        del _pipes[key]
    _pipes.clear()
    gc.collect()
    torch.cuda.empty_cache()
    print('[All models unloaded, VRAM cleared]')

def _get_pipe(key, loader_fn):
    """Get or load a model pipeline."""
    if key not in _pipes:
        unload_all_models()
        loader_fn()
    return _pipes[key]

# --- Scheduler Configuration ---
def _configure_scheduler(pipe, solver='unipc', flow_shift=5.0, use_karras=False,
                          use_dynamic_shifting=False, timestep_spacing='linspace'):
    """Configure the scheduler with advanced parameters."""
    try:
        if solver == 'unipc':
            from diffusers import UniPCMultistepScheduler
            pipe.scheduler = UniPCMultistepScheduler.from_config(
                pipe.scheduler.config, flow_shift=flow_shift,
                use_karras_sigmas=use_karras,
            )
        elif solver == 'euler':
            from diffusers import EulerDiscreteScheduler
            pipe.scheduler = EulerDiscreteScheduler.from_config(
                pipe.scheduler.config, use_karras_sigmas=use_karras,
                timestep_spacing=timestep_spacing,
            )
        elif solver == 'euler_ancestral':
            from diffusers import EulerAncestralDiscreteScheduler
            pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
                pipe.scheduler.config, use_karras_sigmas=use_karras,
            )
        elif solver == 'ddim':
            from diffusers import DDIMScheduler
            pipe.scheduler = DDIMScheduler.from_config(
                pipe.scheduler.config, timestep_spacing=timestep_spacing,
            )
        elif solver == 'flow_match_euler':
            from diffusers import FlowMatchEulerDiscreteScheduler
            pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config(
                pipe.scheduler.config, shift=flow_shift,
                use_dynamic_shifting=use_dynamic_shifting,
                use_karras_sigmas=use_karras,
            )
        elif solver == 'flow_match_heun':
            from diffusers import FlowMatchHeunDiscreteScheduler
            pipe.scheduler = FlowMatchHeunDiscreteScheduler.from_config(
                pipe.scheduler.config, shift=flow_shift,
                use_dynamic_shifting=use_dynamic_shifting,
            )
        elif solver == 'tcd':
            from diffusers import TCDScheduler
            pipe.scheduler = TCDScheduler.from_config(pipe.scheduler.config)
    except Exception as e:
        print(f'[Scheduler config skipped: {e}]')
    return pipe

# --- LTX-Video (Fastest) ---
def _load_ltx():
    from diffusers import LTXPipeline
    print('[Loading LTX-Video model...]')
    _pipes['ltx'] = LTXPipeline.from_pretrained(
        "Lightricks/LTX-Video", torch_dtype=torch.float16, cache_dir='/content/model_cache'
    )
    _pipes['ltx'].to(device)
    _pipes['ltx'].vae.enable_tiling()
    print('[LTX-Video loaded!]')

def get_ltx_pipe():
    return _get_pipe('ltx', _load_ltx)

# --- Wan 2.2 TI2V-5B (Best Quality 720P) ---
def _load_wan22():
    from diffusers import WanPipeline, AutoencoderKLWan
    print('[Loading Wan 2.2 TI2V-5B model...]')
    try:
        _pipes['wan22'] = WanPipeline.from_pretrained(
            "Wan-AI/Wan2.2-TI2V-5B-Diffusers", torch_dtype=torch.bfloat16, cache_dir='/content/model_cache'
        )
    except Exception:
        print('[Wan 2.2 not found, falling back to Wan 2.1 1.3B...]')
        vae = AutoencoderKLWan.from_pretrained(
            "Wan-AI/Wan2.1-T2V-1.3B-Diffusers", subfolder="vae",
            torch_dtype=torch.float32, cache_dir='/content/model_cache'
        )
        _pipes['wan22'] = WanPipeline.from_pretrained(
            "Wan-AI/Wan2.1-T2V-1.3B-Diffusers", vae=vae,
            torch_dtype=torch.float16, cache_dir='/content/model_cache'
        )
    _pipes['wan22'].enable_model_cpu_offload()
    print('[Wan 2.2 loaded!]')

def get_wan22_pipe():
    return _get_pipe('wan22', _load_wan22)

# --- Motif-Video 2B (Balanced, GGUF Support) ---
def _load_motif():
    print('[Loading Motif-Video 2B model...]')
    try:
        from diffusers import MotifVideoPipeline
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3 if device == 'cuda' else 0
        if vram < 16:
            try:
                from diffusers import GGUFQuantizationConfig, MotifVideoTransformer3DModel
                from huggingface_hub import hf_hub_download
                ckpt_path = hf_hub_download("Motif-Technologies/Motif-Video-2B-GGUF", filename="motifv-2b-dev-Q5_K_M.gguf")
                transformer = MotifVideoTransformer3DModel.from_single_file(
                    ckpt_path, quantization_config=GGUFQuantizationConfig(compute_dtype=torch.bfloat16),
                    config="Motif-Technologies/Motif-Video-2B", subfolder="transformer", torch_dtype=torch.bfloat16,
                )
                _pipes['motif'] = MotifVideoPipeline.from_pretrained(
                    "Motif-Technologies/Motif-Video-2B", torch_dtype=torch.bfloat16,
                    transformer=transformer, cache_dir='/content/model_cache'
                )
                print('[Motif-Video 2B loaded with GGUF Q5_K_M!]')
            except Exception as e:
                print(f'[GGUF failed ({e}), trying BF16...]')
                _pipes['motif'] = MotifVideoPipeline.from_pretrained(
                    "Motif-Technologies/Motif-Video-2B", torch_dtype=torch.bfloat16, cache_dir='/content/model_cache'
                )
        else:
            _pipes['motif'] = MotifVideoPipeline.from_pretrained(
                "Motif-Technologies/Motif-Video-2B", torch_dtype=torch.bfloat16, cache_dir='/content/model_cache'
            )
        _pipes['motif'].enable_model_cpu_offload()
    except Exception as e:
        print(f'[Motif unavailable: {e}, falling back to CogVideoX-2B...]')
        from diffusers import CogVideoXPipeline
        _pipes['motif'] = CogVideoXPipeline.from_pretrained("THUDM/CogVideoX-2b", torch_dtype=torch.float16, cache_dir='/content/model_cache')
        _pipes['motif'].enable_model_cpu_offload()
        _pipes['motif'].vae.enable_slicing()
        _pipes['motif'].vae.enable_tiling()
        print('[CogVideoX-2B loaded as fallback!]')

def get_motif_pipe():
    return _get_pipe('motif', _load_motif)

# --- Helios-Distilled (Real-time, Minute-scale) ---
def _load_helios():
    print('[Loading Helios-Distilled model...]')
    try:
        from diffusers import HeliosPyramidPipeline
        _pipes['helios'] = HeliosPyramidPipeline.from_pretrained(
            "BestWishYsh/Helios-Distilled", torch_dtype=torch.bfloat16, cache_dir='/content/model_cache'
        )
        try:
            _pipes['helios'].enable_group_offload(onload_device=device, offload_device="cpu", num_steps=8)
        except Exception:
            _pipes['helios'].enable_model_cpu_offload()
        print('[Helios-Distilled loaded!]')
    except Exception as e:
        print(f'[Helios unavailable: {e}]')
        _pipes['helios'] = None

def get_helios_pipe():
    return _get_pipe('helios', _load_helios)

# --- HoloCine (Multi-shot Narrative) ---
def _load_holocine():
    print('[Loading HoloCine model...]')
    try:
        from diffusers import WanPipeline
        _pipes['holocine'] = WanPipeline.from_pretrained(
            "hlwang06/HoloCine-5B", torch_dtype=torch.bfloat16, cache_dir='/content/model_cache'
        )
        _pipes['holocine'].enable_model_cpu_offload()
        print('[HoloCine loaded!]')
    except Exception as e:
        print(f'[HoloCine unavailable: {e}, falling back to Wan 2.2...]')
        _load_wan22()
        _pipes['holocine'] = _pipes.get('wan22')

def get_holocine_pipe():
    return _get_pipe('holocine', _load_holocine)

# --- Camera prompt injection ---
def _inject_camera_motion(prompt, camera_enabled=False, camera_motion='static',
                          camera_direction=None, camera_speed=0.5, camera_intensity=0.5):
    """Inject camera motion description into the prompt."""
    if not camera_enabled or camera_motion == 'static':
        return prompt
    speed_words = {0.1: 'very slow', 0.3: 'slow', 0.5: 'moderate', 0.7: 'fast', 0.9: 'very fast'}
    speed_str = 'moderate'
    for threshold, word in sorted(speed_words.items()):
        if camera_speed <= threshold:
            speed_str = word
            break
    descriptions = {
        'pan': f'{speed_str} pan {camera_direction or "left"}',
        'tilt': f'{speed_str} tilt {camera_direction or "up"}',
        'zoom': f'{speed_str} zoom {camera_direction or "in"}',
        'dolly': f'{speed_str} dolly {camera_direction or "in"}',
        'dolly_zoom': f'{speed_str} dolly zoom vertigo effect',
        'orbit': f'{speed_str} orbit {camera_direction or "left"}',
        'crane': f'{speed_str} crane {camera_direction or "up"}',
        'pedestal': f'{speed_str} pedestal {camera_direction or "up"}',
        'tracking': f'{speed_str} tracking shot',
        'handheld': 'handheld camera, slight shake',
        'aerial': 'aerial drone shot, sweeping view',
        'custom': 'custom camera movement',
    }
    desc = descriptions.get(camera_motion, '')
    if desc:
        return f'{prompt}, {desc} camera movement'
    return prompt

# --- Generation Functions (with full settings support) ---

def generate_ltx(prompt, negative_prompt, num_frames=97, fps=24, width=768, height=512,
                 steps=30, seed=None, guidance_scale=3.0, guidance_rescale=0.0,
                 decode_timestep=0.03, decode_noise_scale=0.025, image_cond_noise_scale=0.0,
                 solver='unipc', flow_shift=5.0, use_karras=False, custom_timesteps=None,
                 num_videos_per_prompt=1, output_type='pil'):
    '''LTX-Video: Fastest, 768x512, 24fps.'''
    if seed is None: seed = int(time.time())
    pipe = get_ltx_pipe()
    pipe = _configure_scheduler(pipe, solver=solver, flow_shift=flow_shift, use_karras=use_karras)
    generator = torch.Generator(device='cuda').manual_seed(seed)
    kwargs = dict(
        prompt=prompt, negative_prompt=negative_prompt,
        width=width, height=height, num_frames=num_frames,
        num_inference_steps=steps, guidance_scale=guidance_scale,
        guidance_rescale=guidance_rescale,
        decode_timestep=decode_timestep, decode_noise_scale=decode_noise_scale,
        image_cond_noise_scale=image_cond_noise_scale,
        generator=generator, num_videos_per_prompt=num_videos_per_prompt,
        output_type=output_type,
    )
    if custom_timesteps: kwargs['timesteps'] = custom_timesteps
    video = pipe(**kwargs).frames[0]
    output_path = f'/content/outputs/{uuid.uuid4().hex[:8]}.mp4'
    os.makedirs('/content/outputs', exist_ok=True)
    export_to_video(video, output_path, fps=fps)
    return output_path

def generate_wan22(prompt, negative_prompt, num_frames=121, fps=24, width=1280, height=704,
                   steps=50, seed=None, guidance_scale=5.0, guidance_rescale=0.0,
                   solver='unipc', flow_shift=5.0, use_karras=False, use_dynamic_shifting=False,
                   boundary_ratio=0.875, num_videos_per_prompt=1, output_type='pil'):
    '''Wan 2.2 TI2V-5B: 720P, 24fps, best quality.'''
    if seed is None: seed = int(time.time())
    pipe = get_wan22_pipe()
    pipe = _configure_scheduler(pipe, solver=solver, flow_shift=flow_shift, use_karras=use_karras,
                                use_dynamic_shifting=use_dynamic_shifting)
    generator = torch.Generator(device='cuda').manual_seed(seed)
    video = pipe(
        prompt=prompt, negative_prompt=negative_prompt,
        height=height, width=width, num_frames=num_frames,
        num_inference_steps=steps, guidance_scale=guidance_scale,
        generator=generator, num_videos_per_prompt=num_videos_per_prompt,
        output_type=output_type,
    ).frames[0]
    output_path = f'/content/outputs/{uuid.uuid4().hex[:8]}.mp4'
    os.makedirs('/content/outputs', exist_ok=True)
    export_to_video(video, output_path, fps=fps)
    return output_path

def generate_motif(prompt, negative_prompt, num_frames=121, fps=24, width=1280, height=736,
                   steps=50, seed=None, guidance_scale=5.0, guidance_rescale=0.0,
                   solver='unipc', flow_shift=5.0, num_videos_per_prompt=1, output_type='pil'):
    '''Motif-Video 2B: 720P, GGUF quantized, balanced.'''
    if seed is None: seed = int(time.time())
    pipe = get_motif_pipe()
    pipe = _configure_scheduler(pipe, solver=solver, flow_shift=flow_shift)
    generator = torch.Generator(device='cuda').manual_seed(seed)
    if 'MotifVideo' in type(pipe).__name__:
        video = pipe(
            prompt=prompt, negative_prompt=negative_prompt,
            height=height, width=width, num_frames=num_frames,
            num_inference_steps=steps, guidance_scale=guidance_scale,
            generator=generator, num_videos_per_prompt=num_videos_per_prompt,
            output_type=output_type,
        ).frames[0]
    else:
        cog_frames = min(num_frames * fps // 8, 49)
        video = pipe(
            prompt=prompt, num_videos_per_prompt=1,
            num_inference_steps=steps, num_frames=cog_frames,
            guidance_scale=6, generator=generator,
        ).frames[0]
    output_path = f'/content/outputs/{uuid.uuid4().hex[:8]}.mp4'
    os.makedirs('/content/outputs', exist_ok=True)
    export_to_video(video, output_path, fps=fps)
    return output_path

def generate_helios(prompt, negative_prompt, num_frames=240, fps=24, width=832, height=480,
                    steps=3, seed=None, guidance_scale=1.0, output_type='pil'):
    '''Helios-Distilled: Real-time, minute-scale, 3-step distillation.'''
    if seed is None: seed = int(time.time())
    pipe = get_helios_pipe()
    if pipe is None: raise RuntimeError('Helios model not available')
    generator = torch.Generator(device='cuda').manual_seed(seed)
    video = pipe(
        prompt=prompt, num_frames=num_frames, guidance_scale=guidance_scale,
        is_enable_stage2=True, generator=generator, output_type=output_type,
    ).frames[0]
    output_path = f'/content/outputs/{uuid.uuid4().hex[:8]}.mp4'
    os.makedirs('/content/outputs', exist_ok=True)
    export_to_video(video, output_path, fps=fps)
    return output_path

def generate_holocine(prompt, negative_prompt, num_frames=121, fps=24, width=1280, height=704,
                      steps=50, seed=None, guidance_scale=5.0, guidance_rescale=0.0,
                      solver='unipc', flow_shift=5.0, num_videos_per_prompt=1, output_type='pil'):
    '''HoloCine: Multi-shot narrative with persistent consistency.'''
    if seed is None: seed = int(time.time())
    pipe = get_holocine_pipe()
    pipe = _configure_scheduler(pipe, solver=solver, flow_shift=flow_shift)
    generator = torch.Generator(device='cuda').manual_seed(seed)
    video = pipe(
        prompt=prompt, negative_prompt=negative_prompt,
        height=height, width=width, num_frames=num_frames,
        num_inference_steps=steps, guidance_scale=guidance_scale,
        generator=generator, num_videos_per_prompt=num_videos_per_prompt,
        output_type=output_type,
    ).frames[0]
    output_path = f'/content/outputs/{uuid.uuid4().hex[:8]}.mp4'
    os.makedirs('/content/outputs', exist_ok=True)
    export_to_video(video, output_path, fps=fps)
    return output_path

# --- FFmpeg post-processing (color grading + effects + encoding) ---
def apply_ffmpeg_filters(input_path, output_path, color_grading=None, effects=None):
    """Apply color grading and effects via FFmpeg filters."""
    filters = []
    if color_grading:
        c = color_grading
        if c.get('contrast', 0) != 0: filters.append(f"eq=contrast={1.0 + c['contrast']}")
        if c.get('brightness', 0) != 0 or c.get('gamma', 0) != 0:
            filters.append(f"eq=brightness={c.get('brightness', 0):.2f}:gamma={1.0 + c.get('gamma', 0):.2f}")
        if c.get('saturation', 0) != 0: filters.append(f"eq=saturation={1.0 + c['saturation']:.2f}")
        if c.get('temperature', 0) != 0:
            t = c['temperature']
            filters.append(f"colorbalance=rs={t * 0.3}:bs={-t * 0.3}")
        if c.get('tint', 0) != 0: filters.append(f"colorbalance=gs={c['tint'] * 0.3}")
        if c.get('hue', 0) != 0: filters.append(f"hue=h={c['hue'] * 180}")
    if effects:
        e = effects
        if e.get('vignette_enabled'): filters.append(f"vignette=PI/{5 + e.get('vignette_intensity', 0.3) * 10}")
        if e.get('film_grain_enabled'): filters.append(f"noise=alls={int(e.get('film_grain_amount', 0.15) * 100)}:allf=t")
        if e.get('sharpen_enabled'): filters.append(f"unsharp=5:5:{e.get('sharpen_amount', 0.5) * 1.5}:5:5:{e.get('sharpen_amount', 0.5) * 1.5}")
        if e.get('glow_enabled'): filters.append(f"gblur=sigma={e.get('glow_radius', 10)},mix={e.get('glow_intensity', 0.3)}")
        if e.get('bloom_enabled'): filters.append(f"smartblur=lr=1:lt=0.1,eq=brightness={e.get('bloom_intensity', 0.3) * 0.1}")
    if not filters:
        return input_path
    filter_str = ",".join(filters)
    cmd = ['ffmpeg', '-y', '-i', input_path, '-filter:v', filter_str, '-c:a', 'copy', output_path]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        return output_path
    print(f'[FFmpeg filter failed: {result.stderr[:200]}]')
    return input_path

def encode_video(input_path, output_path, codec='h264', crf=23, preset='medium',
                 tune='none', bitrate=None, maxrate=None, bufsize=None,
                 profile='high', pixel_format='yuv420p', audio_codec='aac',
                 audio_bitrate='128k', audio_enabled=False):
    """Encode video with specified codec and quality settings."""
    codec_map = {'h264': 'libx264', 'h265': 'libx265', 'vp9': 'libvpx-vp9', 'av1': 'libsvtav1'}
    encoder = codec_map.get(codec, 'libx264')
    cmd = ['ffmpeg', '-y', '-i', input_path, '-c:v', encoder]
    if bitrate:
        cmd.extend(['-b:v', str(bitrate)])
    else:
        cmd.extend(['-crf', str(crf)])
    if maxrate: cmd.extend(['-maxrate', str(maxrate)])
    if bufsize: cmd.extend(['-bufsize', str(bufsize)])
    cmd.extend(['-preset', preset])
    if tune and tune != 'none': cmd.extend(['-tune', tune])
    if profile and profile != 'auto': cmd.extend(['-profile:v', profile])
    cmd.extend(['-pix_fmt', pixel_format])
    if audio_enabled:
        cmd.extend(['-c:a', audio_codec, '-b:a', audio_bitrate])
    else:
        cmd.extend(['-an'])
    cmd.append(output_path)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        return output_path
    print(f'[FFmpeg encode failed: {result.stderr[:200]}]')
    return input_path

# --- Master Generation Router (with full settings) ---

def generate_video(prompt, model='auto', style='cinematic', num_frames=97, fps=24,
                   steps=30, seed=None, enhance=True, audio=False,
                   upscale=1, interpolate_fps=0,
                   negative_prompt=None, width=None, height=None,
                   guidance_scale=5.0, guidance_rescale=0.0,
                   solver='unipc', flow_shift=5.0, use_karras=False,
                   use_dynamic_shifting=False, timestep_spacing='linspace',
                   custom_timesteps=None, custom_sigmas=None,
                   boundary_ratio=0.875, decode_timestep=0.05,
                   decode_noise_scale=0.025, image_cond_noise_scale=0.0,
                   denoise_strength=1.0, num_videos_per_prompt=1,
                   creativity_scale=0.5, output_type='pil',
                   camera_enabled=False, camera_motion='static',
                   camera_direction=None, camera_speed=0.5, camera_intensity=0.5,
                   camera_fov=60.0, camera_roll=0.0, camera_pitch=0.0, camera_yaw=0.0,
                   motion_intensity=0.5, temporal_smoothing=True, flicker_elimination=True,
                   color_grading=None, effects=None,
                   codec='h264', crf=23, preset='medium', tune='none',
                   bitrate=None, maxrate=None, bufsize=None,
                   profile='high', pixel_format='yuv420p',
                   upscale_model='realesrgan_x2',
                   interpolate_motion_blur=False,
                   tts_text=None, tts_voice='narrator_male',
                   ambient_prompt=None, music_prompt=None,
                   native_audio=False):
    '''Generate video with full pipeline: enhancement -> generation -> post-processing -> encoding.
    
    Supports all settings from the comprehensive settings schema.
    '''
    # Prompt enhancement
    if enhance:
        enhanced_prompt, neg_prompt = enhance_prompt(prompt, style)
    else:
        enhanced_prompt = prompt
        neg_prompt = negative_prompt or NEGATIVE_PROMPT

    # Camera motion injection
    enhanced_prompt = _inject_camera_motion(
        enhanced_prompt, camera_enabled, camera_motion,
        camera_direction, camera_speed, camera_intensity
    )

    # Auto-select model if needed
    if model == 'auto':
        model = 'ltx'

    # Set default resolution per model
    if width is None:
        width = {'ltx': 768, 'wan22': 1280, 'motif': 1280, 'helios': 832, 'holocine': 1280}.get(model, 768)
    if height is None:
        height = {'ltx': 512, 'wan22': 704, 'motif': 736, 'helios': 480, 'holocine': 704}.get(model, 512)

    print(f'[Generating: model={model}, {width}x{height}, {num_frames}f@{fps}fps, {steps} steps, cfg={guidance_scale}]')

    # Route to model
    gen_kwargs = dict(
        prompt=enhanced_prompt, negative_prompt=neg_prompt,
        num_frames=num_frames, fps=fps, steps=steps, seed=seed,
        guidance_scale=guidance_scale, guidance_rescale=guidance_rescale,
        solver=solver, flow_shift=flow_shift, use_karras=use_karras,
        num_videos_per_prompt=num_videos_per_prompt, output_type=output_type,
    )

    if model == 'ltx':
        gen_kwargs.update(decode_timestep=decode_timestep, decode_noise_scale=decode_noise_scale,
                          image_cond_noise_scale=image_cond_noise_scale, width=width, height=height)
        if custom_timesteps: gen_kwargs['custom_timesteps'] = custom_timesteps
        output_path = generate_ltx(**gen_kwargs)
    elif model in ('wan22', 'wan'):
        gen_kwargs.update(use_dynamic_shifting=use_dynamic_shifting, boundary_ratio=boundary_ratio,
                          width=width, height=height)
        output_path = generate_wan22(**gen_kwargs)
    elif model == 'motif':
        gen_kwargs.update(width=width, height=height)
        output_path = generate_motif(**gen_kwargs)
    elif model == 'helios':
        gen_kwargs.update(width=width, height=height)
        output_path = generate_helios(**gen_kwargs)
    elif model == 'holocine':
        gen_kwargs.update(width=width, height=height)
        output_path = generate_holocine(**gen_kwargs)
    else:
        gen_kwargs.update(decode_timestep=decode_timestep, decode_noise_scale=decode_noise_scale,
                          width=width, height=height)
        output_path = generate_ltx(**gen_kwargs)

    print(f'[Video generated: {output_path}]')

    # Post-processing: super-resolution
    if upscale > 1:
        print(f'[Super-resolution {upscale}x...]')
        output_path = post_process_video(output_path, upscale=upscale, target_fps=0)

    # Post-processing: frame interpolation
    if interpolate_fps > 0:
        print(f'[Frame interpolation to {interpolate_fps} FPS...]')
        output_path = post_process_video(output_path, upscale=1, target_fps=interpolate_fps)

    # Post-processing: color grading + effects via FFmpeg
    if color_grading or effects:
        filtered_path = output_path.replace('.mp4', '_graded.mp4')
        output_path = apply_ffmpeg_filters(output_path, filtered_path, color_grading, effects)

    # Final encoding
    if codec != 'h264' or crf != 23 or preset != 'medium' or tune != 'none' or bitrate:
        encoded_path = output_path.replace('.mp4', f'_{codec}.mp4')
        output_path = encode_video(output_path, encoded_path, codec=codec, crf=crf,
                                   preset=preset, tune=tune, bitrate=bitrate,
                                   maxrate=maxrate, bufsize=bufsize, profile=profile,
                                   pixel_format=pixel_format, audio_enabled=audio)

    return output_path

print('Video generation functions ready (full settings support)!')
print('Models load on-demand (one at a time) to prevent RAM crashes')
print('\nAvailable models:')
print('  ltx      -> LTX-Video (fastest, 768x512)')
print('  wan22    -> Wan 2.2 TI2V-5B (720P, best quality)')
print('  motif    -> Motif-Video 2B (720P, GGUF quantized)')
print('  helios   -> Helios-Distilled (real-time, minute-scale)')
print('  holocine -> HoloCine (multi-shot narrative)')
print('\nFull settings: scheduler, camera, color grading, effects, encoding, audio')

## Step 5: Post-Processing Pipeline

AI video enhancement: Super-resolution upscaling (Real-ESRGAN) + Frame interpolation (RIFE-style).

In [ ]:
import subprocess
import tempfile

def post_process_video(input_path, upscale=1, target_fps=0):
    """Apply post-processing: super-resolution + frame interpolation.
    
    Args:
        input_path: Path to input video
        upscale: Super-resolution factor (1=skip, 2, 4)
        target_fps: Target FPS for interpolation (0=skip)
    Returns:
        Path to enhanced video
    """
    if upscale <= 1 and target_fps <= 0:
        return input_path
    
    output_path = input_path.replace('.mp4', '_enhanced.mp4')
    
    if upscale > 1:
        print(f'[Super-resolution {upscale}x with Real-ESRGAN...]')
        try:
            from realesrgan import RealESRGANer
            from basicsr.archs.rrdbnet_arch import RRDBNet
            import cv2
            
            model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                          num_block=23, num_grow_ch=32, scale=upscale)
            upsampler = RealESRGANer(
                scale=upscale, model_path=None,
                model=model, tile=0, tile_pad=10, pre_pad=0,
                half=True if device == 'cuda' else False,
            )
            
            cap = cv2.VideoCapture(input_path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_path, fourcc, fps, (w * upscale, h * upscale))
            
            frame_idx = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                enhanced, _ = upsampler.enhance(frame, outscale=upscale)
                out.write(enhanced)
                frame_idx += 1
                if frame_idx % 10 == 0:
                    print(f'  [SR: {frame_idx}/{total} frames]')
            
            cap.release()
            out.release()
            print(f'[Super-resolution complete: {output_path}]')
            input_path = output_path
        except Exception as e:
            print(f'[Super-resolution failed: {e}]')
            print('[Skipping upscaling, continuing with original...]')
    
    if target_fps > 0:
        print(f'[Frame interpolation to {target_fps} FPS...]')
        try:
            interp_path = output_path.replace('_enhanced', f'_{target_fps}fps')
            cmd = [
                'ffmpeg', '-i', input_path,
                '-filter:v', f'minterpolate=fps={target_fps}:mi_mode=mci:mc_mode=aobmc:me_mode=bidir',
                '-c:a', 'copy',
                interp_path
            ]
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode == 0:
                print(f'[Frame interpolation complete: {interp_path}]')
                return interp_path
            else:
                print(f'[ffmpeg interpolation failed: {result.stderr[:200]}]')
        except Exception as e:
            print(f'[Frame interpolation failed: {e}]')
    
    return output_path

print('Post-processing pipeline ready!')
print('  Super-resolution: Real-ESRGAN (2x/4x upscaling)')
print('  Frame interpolation: ffmpeg minterpolate (FPS multiplication)')

## Step 5b: Image Generation (SDXL - RAM-Safe)

Text-to-image generation for posters, character assets, and scene stills.
Uses SDXL (lighter than Flux) to avoid Colab RAM crashes.

In [ ]:
import torch
import os
import time
import uuid
import gc
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
_img_pipe = None

def get_img_pipe():
    global _img_pipe
    if _img_pipe is None:
        from diffusers import StableDiffusionXLPipeline
        print('[Loading SDXL model...]')
        _img_pipe = StableDiffusionXLPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,
            cache_dir='/content/model_cache'
        )
        _img_pipe.to(device)
        print('[SDXL loaded!]')
    return _img_pipe

def unload_img_pipe():
    global _img_pipe
    if _img_pipe is not None:
        del _img_pipe
        _img_pipe = None
        gc.collect()
        torch.cuda.empty_cache()
        print('[Image model unloaded, VRAM cleared]')

def generate_image(prompt, negative_prompt=None, width=1024, height=1024,
                   steps=25, seed=None, guidance_scale=7.5, batch_count=1,
                   style_preset='None'):
    if seed is None:
        seed = int(time.time())
    pipe = get_img_pipe()
    generator = torch.Generator(device='cuda').manual_seed(seed)
    
    style_map = {
        'cinematic': 'cinematic lighting, film still, dramatic shadows, professional cinematography',
        'realistic': 'photorealistic, ultra realistic, 8k, professional photography, natural lighting',
        'anime': 'anime style, cel shaded, vibrant colors, detailed background, studio quality',
        'digital art': 'digital art, concept art, trending on artstation, highly detailed',
        'oil painting': 'oil painting, textured brushstrokes, fine art, museum quality',
        'poster': 'movie poster style, dramatic composition, bold typography area, cinematic lighting, professional poster design',
        'portrait': 'portrait photography, studio lighting, shallow depth of field, professional headshot',
        'fantasy': 'fantasy art, magical atmosphere, ethereal lighting, detailed concept art',
        'sci-fi': 'science fiction, futuristic, neon lighting, cyberpunk aesthetic, high tech',
        'noir': 'film noir, black and white, high contrast, dramatic shadows, vintage',
    }
    suffix = style_map.get(style_preset, '')
    full_prompt = f"{prompt}. {suffix}" if suffix else prompt
    neg = negative_prompt or "worst quality, low quality, blurry, distorted, deformed, ugly, bad anatomy, watermark, text"
    
    print(f'[Image gen: {width}x{height}, {steps} steps, cfg={guidance_scale}]')
    images = pipe(
        prompt=full_prompt, negative_prompt=neg,
        width=width, height=height,
        num_inference_steps=steps, guidance_scale=guidance_scale,
        generator=generator, num_images_per_prompt=batch_count,
    ).images
    
    output_paths = []
    os.makedirs('/content/outputs/images', exist_ok=True)
    for img in images:
        path = f'/content/outputs/images/{uuid.uuid4().hex[:8]}.png'
        img.save(path)
        output_paths.append(path)
        print(f'[Image saved: {path}]')
    return output_paths

print('Image generation ready (SDXL - RAM-safe)!')
print('  sdxl -> Stable Diffusion XL (1024x1024, ~5GB VRAM)')
print('  Image model auto-unloads before video generation to prevent crashes')

## Step 6: Start FastAPI Server with Tunnel

In [ ]:
import threading
import uvicorn
import subprocess
import json
import urllib.request
import nest_asyncio
from fastapi import FastAPI, UploadFile, File, BackgroundTasks
from fastapi.responses import FileResponse, JSONResponse, HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, List, Dict, Any

nest_asyncio.apply()

app = FastAPI(title='SoulIllusions AI Video Maker 2026-2027')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

generation_status = {}
image_status = {}

class GenRequest(BaseModel):
    prompt: str
    model: str = 'auto'
    style: str = 'cinematic'
    num_frames: int = 97
    fps: int = 24
    steps: int = 30
    seed: Optional[int] = None
    enhance: bool = True
    audio: bool = False
    upscale: int = 1
    interpolate_fps: int = 0
    negative_prompt: Optional[str] = None
    width: Optional[int] = None
    height: Optional[int] = None
    guidance_scale: float = 5.0
    guidance_rescale: float = 0.0
    solver: str = 'unipc'
    flow_shift: float = 5.0
    use_karras_sigmas: bool = False
    use_dynamic_shifting: bool = False
    timestep_spacing: str = 'linspace'
    custom_timesteps: Optional[List[float]] = None
    custom_sigmas: Optional[List[float]] = None
    boundary_ratio: float = 0.875
    decode_timestep: float = 0.05
    decode_noise_scale: float = 0.025
    image_cond_noise_scale: float = 0.0
    denoise_strength: float = 1.0
    num_videos_per_prompt: int = 1
    creativity_scale: float = 0.5
    output_type: str = 'pil'
    camera_enabled: bool = False
    camera_motion: str = 'static'
    camera_direction: Optional[str] = None
    camera_speed: float = 0.5
    camera_intensity: float = 0.5
    camera_fov: float = 60.0
    camera_roll: float = 0.0
    camera_pitch: float = 0.0
    camera_yaw: float = 0.0
    motion_intensity: float = 0.5
    temporal_smoothing: bool = True
    flicker_elimination: bool = True
    upscale_model: str = 'realesrgan_x2'
    interpolate_motion_blur: bool = False
    color_grading: Optional[Dict[str, float]] = None
    effects: Optional[Dict[str, Any]] = None
    codec: str = 'h264'
    crf: int = 23
    preset: str = 'medium'
    tune: str = 'none'
    bitrate: Optional[str] = None
    maxrate: Optional[str] = None
    bufsize: Optional[str] = None
    profile: str = 'high'
    pixel_format: str = 'yuv420p'
    native_audio: bool = False
    tts_text: Optional[str] = None
    tts_voice: str = 'narrator_male'
    ambient_prompt: Optional[str] = None
    music_prompt: Optional[str] = None

class ImageGenRequest(BaseModel):
    prompt: str = ""
    model: str = "sdxl"
    negative_prompt: Optional[str] = None
    aspect_ratio: str = "1:1"
    quality: str = "standard"
    seed: Optional[int] = None
    batch_count: int = 1
    style_preset: str = "None"
    width: Optional[int] = None
    height: Optional[int] = None
    guidance_scale: float = 7.5
    steps: int = 25
    lora_model: Optional[str] = None
    lora_weight: float = 1.0
    reference_strength: int = 50
    image_mode: str = "t2i"
    reference_images: list = []

@app.get('/')
async def root():
    return {
        'status': 'online', 'service': 'SoulIllusions AI Video Maker 2026-2027',
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'version': '3.1',
    }

@app.get('/api/status')
async def status():
    return {
        'status': 'online',
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'vram_total': f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB' if torch.cuda.is_available() else 'N/A',
        'vram_free': f'{torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB' if torch.cuda.is_available() else 'N/A',
        'models': ['ltx-video', 'wan-2.2-ti2v-5b', 'motif-video-2b', 'helios-distilled', 'holocine'],
        'features': ['prompt-enhancement', 'super-resolution', 'frame-interpolation', 'gguf-quantization',
                     'camera-control', 'color-grading', 'effects', 'custom-encoding', 'scheduler-config',
                     'custom-timesteps', 'audio-ready', 'image-generation'],
        'queue': len([k for k, v in generation_status.items() if v['status'] == 'processing']),
    }

# === Image Generation Endpoints ===

@app.get('/api/image/options')
async def image_options():
    return {
        'models': [
            {'id': 'sdxl', 'name': 'Stable Diffusion XL', 'desc': 'Versatile (1024x1024)', 'max_resolution': 2048},
        ],
        'style_presets': ['None', 'cinematic', 'realistic', 'anime', 'digital art', 'oil painting',
                          'poster', 'portrait', 'fantasy', 'sci-fi', 'noir'],
        'aspect_ratios': {
            '1:1': {'width': 1024, 'height': 1024},
            '16:9': {'width': 1344, 'height': 768},
            '9:16': {'width': 768, 'height': 1344},
            '4:3': {'width': 1152, 'height': 896},
            '3:4': {'width': 896, 'height': 1152},
            '2:3': {'width': 832, 'height': 1216},
            '3:2': {'width': 1216, 'height': 832},
            '21:9': {'width': 1536, 'height': 640},
        },
        'quality_modes': {
            'draft': {'steps': 10, 'guidance': 3.0},
            'standard': {'steps': 25, 'guidance': 7.5},
            'pro': {'steps': 40, 'guidance': 7.5},
            'ultra': {'steps': 60, 'guidance': 8.0},
        },
    }

@app.post('/api/image/generate')
async def generate_image_api(req: ImageGenRequest, background_tasks: BackgroundTasks):
    job_id = uuid.uuid4().hex[:12]
    image_status[job_id] = {
        'status': 'processing', 'progress': 0,
        'prompt': req.prompt, 'model': req.model, 'images': [],
    }
    aspect_map = {
        '1:1': (1024, 1024), '16:9': (1344, 768), '9:16': (768, 1344),
        '4:3': (1152, 896), '3:4': (896, 1152), '2:3': (832, 1216),
        '3:2': (1216, 832), '21:9': (1536, 640),
    }
    w, h = aspect_map.get(req.aspect_ratio, (1024, 1024))
    if req.width: w = req.width
    if req.height: h = req.height
    quality_steps = {'draft': 10, 'standard': 25, 'pro': 40, 'ultra': 60}
    steps = quality_steps.get(req.quality, req.steps)

    def run_img_gen():
        try:
            paths = generate_image(
                prompt=req.prompt,
                negative_prompt=req.negative_prompt,
                width=w, height=h, steps=steps, seed=req.seed,
                guidance_scale=req.guidance_scale, batch_count=req.batch_count,
                style_preset=req.style_preset,
            )
            image_status[job_id] = {
                'status': 'complete', 'progress': 1.0,
                'prompt': req.prompt, 'model': req.model, 'images': paths,
            }
        except Exception as e:
            image_status[job_id] = {
                'status': 'failed', 'progress': 0,
                'prompt': req.prompt, 'images': [], 'error': str(e),
            }

    background_tasks.add_task(run_img_gen)
    return {'job_id': job_id, 'status': 'processing', 'model': req.model}

@app.get('/api/image/status/{job_id}')
async def image_job_status(job_id: str):
    if job_id not in image_status:
        return JSONResponse({'error': 'Job not found'}, status_code=404)
    return image_status[job_id]

@app.get('/api/image/download/{job_id}')
async def image_download(job_id: str):
    if job_id not in image_status:
        return JSONResponse({'error': 'Job not found'}, status_code=404)
    job = image_status[job_id]
    if job['status'] != 'complete' or not job['images']:
        return JSONResponse({'error': 'Image not ready'}, status_code=400)
    return FileResponse(job['images'][0], media_type='image/png', filename=f'soulillusions_{job_id}.png')

# === Video Generation Endpoints ===

@app.post('/api/generate')
async def generate(req: GenRequest, background_tasks: BackgroundTasks):
    job_id = uuid.uuid4().hex[:12]
    generation_status[job_id] = {
        'status': 'processing', 'progress': 0,
        'prompt': req.prompt, 'model': req.model, 'output': None,
    }
    
    def run_gen():
        try:
            unload_img_pipe()
            output_path = generate_video(
                prompt=req.prompt, model=req.model, style=req.style,
                num_frames=req.num_frames, fps=req.fps, steps=req.steps,
                seed=req.seed, enhance=req.enhance, audio=req.audio,
                upscale=req.upscale, interpolate_fps=req.interpolate_fps,
                negative_prompt=req.negative_prompt, width=req.width, height=req.height,
                guidance_scale=req.guidance_scale, guidance_rescale=req.guidance_rescale,
                solver=req.solver, flow_shift=req.flow_shift, use_karras=req.use_karras_sigmas,
                use_dynamic_shifting=req.use_dynamic_shifting, timestep_spacing=req.timestep_spacing,
                custom_timesteps=req.custom_timesteps, custom_sigmas=req.custom_sigmas,
                boundary_ratio=req.boundary_ratio, decode_timestep=req.decode_timestep,
                decode_noise_scale=req.decode_noise_scale, image_cond_noise_scale=req.image_cond_noise_scale,
                denoise_strength=req.denoise_strength, num_videos_per_prompt=req.num_videos_per_prompt,
                creativity_scale=req.creativity_scale, output_type=req.output_type,
                camera_enabled=req.camera_enabled, camera_motion=req.camera_motion,
                camera_direction=req.camera_direction, camera_speed=req.camera_speed,
                camera_intensity=req.camera_intensity, camera_fov=req.camera_fov,
                camera_roll=req.camera_roll, camera_pitch=req.camera_pitch, camera_yaw=req.camera_yaw,
                motion_intensity=req.motion_intensity, temporal_smoothing=req.temporal_smoothing,
                flicker_elimination=req.flicker_elimination,
                color_grading=req.color_grading, effects=req.effects,
                codec=req.codec, crf=req.crf, preset=req.preset, tune=req.tune,
                bitrate=req.bitrate, maxrate=req.maxrate, bufsize=req.bufsize,
                profile=req.profile, pixel_format=req.pixel_format,
                upscale_model=req.upscale_model, interpolate_motion_blur=req.interpolate_motion_blur,
                tts_text=req.tts_text, tts_voice=req.tts_voice,
                ambient_prompt=req.ambient_prompt, music_prompt=req.music_prompt,
                native_audio=req.native_audio,
            )
            generation_status[job_id] = {
                'status': 'complete', 'progress': 1.0,
                'prompt': req.prompt, 'output': output_path,
            }
        except Exception as e:
            generation_status[job_id] = {
                'status': 'failed', 'progress': 0,
                'prompt': req.prompt, 'output': None, 'error': str(e),
            }
    
    background_tasks.add_task(run_gen)
    return {'job_id': job_id, 'status': 'processing', 'model': req.model}

@app.get('/api/status/{job_id}')
async def job_status(job_id: str):
    if job_id not in generation_status:
        return JSONResponse({'error': 'Job not found'}, status_code=404)
    return generation_status[job_id]

@app.get('/api/download/{job_id}')
async def download(job_id: str):
    if job_id not in generation_status:
        return JSONResponse({'error': 'Job not found'}, status_code=404)
    job = generation_status[job_id]
    if job['status'] != 'complete' or not job['output']:
        return JSONResponse({'error': 'Video not ready'}, status_code=400)
    return FileResponse(job['output'], media_type='video/mp4', filename=f'soulillusions_{job_id}.mp4')

@app.get('/api/styles')
async def styles():
    return {'styles': list(STYLE_MODIFIERS.keys())}

@app.get('/api/models')
async def models():
    return {
        'models': [
            {'id': 'ltx', 'name': 'LTX-Video', 'desc': 'Fastest (768x512, 24fps, ~30-60s)', 'resolution': '768x512', 'fps': 24},
            {'id': 'wan22', 'name': 'Wan 2.2 TI2V-5B', 'desc': 'Best quality 720P (1280x704, 24fps)', 'resolution': '1280x704', 'fps': 24},
            {'id': 'motif', 'name': 'Motif-Video 2B', 'desc': 'Balanced 720P GGUF (1280x736, 24fps)', 'resolution': '1280x736', 'fps': 24},
            {'id': 'helios', 'name': 'Helios-Distilled', 'desc': 'Real-time minute-scale (832x480, 24fps)', 'resolution': '832x480', 'fps': 24},
            {'id': 'holocine', 'name': 'HoloCine', 'desc': 'Multi-shot narrative (1280x704, 24fps)', 'resolution': '1280x704', 'fps': 24},
        ]
    }

@app.post('/api/enhance-prompt')
async def enhance_prompt_api(prompt: str, style: str = 'cinematic'):
    enhanced, negative = enhance_prompt(prompt, style)
    return {'original': prompt, 'enhanced': enhanced, 'negative_prompt': negative, 'style': style}

@app.post('/api/post-process')
async def post_process_api(job_id: str, upscale: int = 2, interpolate_fps: int = 0):
    if job_id not in generation_status:
        return JSONResponse({'error': 'Job not found'}, status_code=404)
    job = generation_status[job_id]
    if job['status'] != 'complete' or not job['output']:
        return JSONResponse({'error': 'Video not ready'}, status_code=400)
    try:
        enhanced_path = post_process_video(job['output'], upscale=upscale, target_fps=interpolate_fps)
        return {'status': 'complete', 'output': enhanced_path}
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

@app.get('/api/settings/defaults')
async def settings_defaults():
    return {
        'aspect_ratios': {
            '16:9': {'width': 1280, 'height': 720}, '9:16': {'width': 720, 'height': 1280},
            '1:1': {'width': 1024, 'height': 1024}, '4:3': {'width': 1024, 'height': 768},
            '21:9': {'width': 1280, 'height': 544}, '2.39:1': {'width': 1280, 'height': 536},
            '4:5': {'width': 896, 'height': 1120},
        },
        'quality_modes': {
            'draft': {'steps': 10, 'guidance': 3.0}, 'standard': {'steps': 30, 'guidance': 5.0},
            'pro': {'steps': 50, 'guidance': 5.0}, 'turbo': {'steps': 5, 'guidance': 1.0},
            'ultra': {'steps': 80, 'guidance': 6.0},
        },
        'camera_presets': ['static', 'pan_left', 'pan_right', 'tilt_up', 'tilt_down', 'zoom_in',
                          'zoom_out', 'dolly_in', 'dolly_out', 'dolly_zoom', 'orbit_left',
                          'orbit_right', 'crane_up', 'crane_down', 'tracking', 'handheld', 'aerial'],
        'schedulers': ['unipc', 'euler', 'euler_ancestral', 'ddim', 'dpm_plus_plus',
                       'flow_match_euler', 'flow_match_heun', 'tcd'],
        'codecs': ['h264', 'h265', 'vp9', 'av1'],
        'tune_options': ['none', 'film', 'animation', 'stillimage', 'fastdecode', 'zerolatency'],
        'styles': list(STYLE_MODIFIERS.keys()),
    }

# Start server in background thread
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print('Server started on port 8000')

time.sleep(3)
subprocess.run(['pkill', '-f', 'lt'], capture_output=True)
lt_process = subprocess.Popen(['lt', '--port', '8000'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)
lt_output = lt_process.stdout.read1(4096).decode('utf-8', errors='ignore')

import re
url_match = re.search(r'https://[a-zA-Z0-9-]+\.loca\.lt', lt_output)

if url_match:
    public_url = url_match.group(0)
else:
    try:
        response = urllib.request.urlopen('http://localhost:8000', timeout=5)
        public_url = 'Check localtunnel output below for your URL'
    except:
        public_url = 'Check localtunnel output below for your URL'
    print(f'localtunnel output: {lt_output}')

print(f'\n{"="*60}')
print(f'  SoulIllusions AI Video Maker 2026-2027 v3.1 - GPU Backend LIVE!')
print(f'  Public URL: {public_url}')
print(f'  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'  Video: LTX + Wan 2.2 + Motif 2B + Helios + HoloCine')
print(f'  Image: SDXL (RAM-safe)')
print(f'  Full Settings: Camera + Color Grading + Effects + Encoding + Scheduler')
print(f'{"="*60}')
print(f'\n  Copy this URL and paste it into the SoulIllusions desktop app:')
print(f'  {public_url}')
print(f'\n  Server is running! Keep this notebook open.')

## Step 7: Quick Test (Optional)
Run this to test that video generation works before using the desktop app.

In [ ]:
# Quick test - generate a short video using LTX-Video (fastest model)
print('Testing LTX-Video generation (fastest model)...')
test_path = generate_video(
    'A cat walking on a beach at sunset, cinematic golden hour lighting',
    model='ltx',
    style='cinematic',
    num_frames=97,
    fps=24,
    steps=30,
)
print(f'Test video saved: {test_path}')
print(f'File size: {os.path.getsize(test_path) / 1024:.0f} KB')
print('Test complete! LTX-Video is working.')
print('\nTo test other models:')
print("  generate_video('your prompt', model='wan22', style='cinematic')  # 720P best quality")
print("  generate_video('your prompt', model='motif', style='cinematic')  # 720P GGUF balanced")
print("  generate_video('your prompt', model='helios', style='cinematic')  # Real-time, minute-scale")
print("  generate_video('your prompt', model='holocine', style='cinematic')  # Multi-shot narrative")
print('\nTo test post-processing:')
print("  generate_video('your prompt', model='ltx', upscale=2)  # 2x super-resolution")
print("  generate_video('your prompt', model='ltx', interpolate_fps=60)  # Interpolate to 60fps")
print('\nTo test prompt enhancement:')
print("  enhance_prompt('A dog running in a park', 'cinematic')  # Returns (enhanced, negative)")